In [1]:
import json 
try: 
    with open('.\\Spider_data\\spider_data\\train_spider.json','r',encoding='utf8') as f: 
        lines = json.load(f)
        q_records = lines
except Exception: 
    print("Lỗi!")

try: 
    with open('.\\Spider_data\\spider_data\\tables.json','r',encoding='utf8') as f: 
        lines = json.load(f)
        db_records = lines
except Exception: 
    print("Lỗi!")

try: 
    with open('.\\Spider_data\\spider_data\\dev.json',encoding='utf8') as f:
        dev_json_contents = json.load(f)
except Exception:
    print("Lỗi")
    
#lọc ra danh sách các cột của bảng dưới dạng dict {(STT, tên cột): [(vị trí cột, tên cột)]} VD: {(0,head),[(1,headid),(2,name)]}
def table_names_Mapping(db_inf: dict):
    columns = db_inf['column_names']
    tables = db_inf['table_names']
    n_d = {}
    for i,v in enumerate(tables):
        s_l = []
        for j,l in enumerate(columns):
            if i == l[0]:
                s_l.append((j,l[1]))
        n_d[(i,v)] = s_l
    return n_d

def foreign_key_mapping(d: dict): 
    l = d['foreign_keys'] 
    columns = d['column_names']
    tables = d['table_names']
    s = []
    for i in l:
        fk_i = i[0]   
        pk_i = i[1]
        fk_column = columns[fk_i][1]
        pk_column = columns[pk_i][1]
        t_i1 = columns[fk_i][0] 
        t_i2 = columns[pk_i][0] 
        table_fk = tables[t_i1]
        table_pk = tables[t_i2]
        s.append(table_fk + "." + fk_column + " khóa ngoại của " + table_pk + "." + pk_column)
    return s

def table_names_original_mapping(db_inf: dict):
    columns = db_inf['column_names_original']
    tables = db_inf['table_names_original']
    n_d = {}
    for i,v in enumerate(tables):
        s_l = []
        for j,l in enumerate(columns):
            if i == l[0]:
                s_l.append((j,l[1]))
        n_d[(i,v)] = s_l
    return n_d

#model code T5 small 60 triệu tham số
def extract_table_names_orginal(d: dict):
    return d['table_names_original']


def get_question(d: dict):
    return d['question']

def build_schema_text(db_inf: dict) -> str: 
    tables = db_inf['table_names_original']
    columns = db_inf['column_names_original']
    foreign_key = db_inf['foreign_keys']
    schema_lines = []
    schema_lines.append('Tables info: ')
    for table_indx, table_name in enumerate(tables):
        table_columns = [
            column_name
            for table_indx_in_columns,column_name in columns
            if table_indx_in_columns == table_indx
        ]
        schema_lines.append(f"{table_name}({', '.join((table_columns))})")
    
    relation_lines = []

    for column_indx1, column_indx2 in foreign_key:
        table_indx1, col_name1 = columns[column_indx1]
        table_indx2, col_name2 = columns[column_indx2]
        if table_indx1 >= 0 and table_indx2 >= 0:
            relation_lines.append(
                f"{tables[table_indx1]}.{col_name1}"
                f"<-> {tables[table_indx2]}.{col_name2}"
            )
    if relation_lines:
        schema_lines.append('Column relations: ')
        schema_lines.extend(relation_lines)
    return "\n".join(schema_lines)



In [2]:
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq
model_name = 'google-t5/t5-small'

#load tokenizer của model t5 có checkpoint name là 'google-t5/t5-small'
tokenizer_t5 = AutoTokenizer.from_pretrained(model_name)
#load model t5 có checkpoint là 'google-t5/t5-small'
model = AutoModelForSeq2SeqLM.from_pretrained(model_name) 

#Tạo dữ liệu đầu vào 
training_samples = []
for i in q_records:
    db_id = i['db_id']
    for db in db_records: 
        if db['db_id'] == db_id:
            db_inf = db
            break
    schema_text = build_schema_text(db_inf)
    input_text = (
        f"Question: {i['question']}\n"
        f"Schema: {schema_text}"
    )
    training_samples.append({"input_text": input_text,
                             "output": i['query']})

#xây dựng hàm tokenize theo batch dữ liệu (ví dụ mỗi batch là 128 dữ liệu)
def tokenize_batch(batch):
    model_inputs = tokenizer_t5(batch['input_text'],
                               truncation=True,
                               max_length=640)

    sql_query_encode = tokenizer_t5(batch['output'],
                                    truncation=True,
                                    max_length=256)

    model_inputs['labels'] = sql_query_encode['input_ids']
    return model_inputs

#chuyển danh sách dữ liệu sang dataset để dễ token cũng như thực hiện thao tác
raw_dataset = Dataset.from_list(training_samples)
tokenized_dataset = raw_dataset.map(
    tokenize_batch,
    batched=True,
    batch_size=128,
    remove_columns=raw_dataset.column_names
)

validation_samples = []
for i in dev_json_contents:
    db_id = i['db_id']
    for db in db_records:
        if db['db_id'] == db_id:
            db_inf = db
            break
    input_text = (
        f"Question: {i['question']}\n"
        f"Schema: {build_schema_text(db_inf)}"
    )
    validation_samples.append({
        "input_text": input_text,
        "output": i['query']
    })

#chuyển danh sách dữ liệu tập validation thành dataset
raw_validation_set = Dataset.from_list(validation_samples)

#tokenizer từng batch, thu được dict với mỗi key lưu trữ value là 1 danh sách các batch, ví dụ 3 batch mỗi batch 128 dữ liệu thì sẽ là 3 danh sách 128 phần tử 
tokenizer_validation_set = raw_validation_set.map(
    tokenize_batch,
    batched=True,
    batch_size=128,
    remove_columns=raw_validation_set.column_names
)
print(tokenizer_validation_set[:2])


data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer_t5,model=model)
sample_batch = data_collator([tokenized_dataset[i] for i in range(4)])


# chọn vị trí chạy của model (GPU hoặc CPU)
device = torch.device('cuda' if torch.cuda.is_available() else "cpu")

d:\NLP_Projects\NLP_tieu_luan\.venv_NLP\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Map: 100%|██████████| 1034/1034 [00:00<00:00, 9007.33 examples/s]


{'input_ids': [[11860, 10, 571, 186, 7634, 7, 103, 62, 43, 58, 10248, 51, 9, 10, 4398, 7, 2845, 10, 14939, 599, 134, 17, 9, 12925, 834, 4309, 6, 10450, 6, 5570, 6, 4000, 9, 6726, 6, 1592, 222, 6, 5586, 222, 6, 23836, 61, 7634, 599, 134, 9963, 834, 4309, 6, 5570, 6, 6993, 6, 11263, 834, 23954, 6, 11263, 834, 21019, 834, 1201, 6, 7526, 6, 27, 7, 834, 13513, 61, 4219, 599, 11620, 52, 17, 834, 4309, 6, 4219, 834, 23954, 6, 37, 526, 6, 12750, 834, 4309, 6, 2929, 61, 7634, 834, 77, 834, 11620, 52, 17, 599, 11620, 52, 17, 834, 4309, 6, 24366, 834, 4309, 61, 29926, 5836, 10, 4219, 5, 134, 17, 9, 12925, 834, 4309, 2, 13114, 14939, 5, 134, 17, 9, 12925, 834, 4309, 7634, 834, 77, 834, 11620, 52, 17, 5, 134, 9963, 834, 4309, 2, 13114, 7634, 5, 134, 9963, 834, 4309, 7634, 834, 77, 834, 11620, 52, 17, 5, 11620, 52, 17, 834, 4309, 2, 13114, 4219, 5, 11620, 52, 17, 834, 4309, 1], [11860, 10, 363, 19, 8, 792, 381, 13, 7634, 7, 58, 10248, 51, 9, 10, 4398, 7, 2845, 10, 14939, 599, 134, 17, 9, 12925, 834,

In [3]:
db_in4 = {
      "column_names": [
         [
            -1,
            "*"
         ],
         [
            0,
            "singer id"
         ],
         [
            0,
            "name"
         ],
         [
            0,
            "birth year"
         ],
         [
            0,
            "net worth millions"
         ],
         [
            0,
            "citizenship"
         ],
         [
            1,
            "song id"
         ],
         [
            1,
            "title"
         ],
         [
            1,
            "singer id"
         ],
         [
            1,
            "sales"
         ],
         [
            1,
            "highest position"
         ]
      ],
      "column_names_original": [
         [
            -1,
            "*"
         ],
         [
            0,
            "Singer_ID"
         ],
         [
            0,
            "Name"
         ],
         [
            0,
            "Birth_Year"
         ],
         [
            0,
            "Net_Worth_Millions"
         ],
         [
            0,
            "Citizenship"
         ],
         [
            1,
            "Song_ID"
         ],
         [
            1,
            "Title"
         ],
         [
            1,
            "Singer_ID"
         ],
         [
            1,
            "Sales"
         ],
         [
            1,
            "Highest_Position"
         ]
      ],
      "column_types": [
         "text",
         "number",
         "text",
         "number",
         "number",
         "text",
         "number",
         "text",
         "number",
         "number",
         "number"
      ],
      "db_id": "singer",
      "foreign_keys": [
         [
            8,
            1
         ]
      ],
      "primary_keys": [
         1,
         6
      ],
      "table_names": [
         "singer",
         "song"
      ],
      "table_names_original": [
         "singer",
         "song"
      ]
   }

schema = build_schema_text(db_in4)
print(f"Schema")
print(schema)
print(input_text)
print(len(raw_dataset))
print(len(raw_validation_set))

Schema
Tables info: 
singer(Singer_ID, Name, Birth_Year, Net_Worth_Millions, Citizenship)
song(Song_ID, Title, Singer_ID, Sales, Highest_Position)
Column relations: 
song.Singer_ID<-> singer.Singer_ID
Question: What are the names of properties that are either houses or apartments with more than 1 room?
Schema: Tables info: 
Ref_Feature_Types(feature_type_code, feature_type_name)
Ref_Property_Types(property_type_code, property_type_description)
Other_Available_Features(feature_id, feature_type_code, feature_name, feature_description)
Properties(property_id, property_type_code, date_on_market, date_sold, property_name, property_address, room_count, vendor_requested_price, buyer_offered_price, agreed_selling_price, apt_feature_1, apt_feature_2, apt_feature_3, fld_feature_1, fld_feature_2, fld_feature_3, hse_feature_1, hse_feature_2, hse_feature_3, oth_feature_1, oth_feature_2, oth_feature_3, shp_feature_1, shp_feature_2, shp_feature_3, other_property_details)
Other_Property_Features(prope

In [5]:
for i in range(4):
    print(f"\n Mẫu {i + 1}")

    input_text = tokenizer_t5.decode(
        sample_batch["input_ids"][i],
        skip_special_tokens=True
    )

    valid_labels = sample_batch['labels'][i][
        sample_batch['labels'][i] != -100
    ]

    target_sql = tokenizer_t5.decode(
        valid_labels,
        skip_special_tokens=True
    )

    # print("INPUT: ")
    # print(input_text)

    print("\nSQL TARGET: ")
    print(target_sql)


 Mẫu 1

SQL TARGET: 
SELECT count(*) FROM head WHERE age > 56

 Mẫu 2

SQL TARGET: 
SELECT name, born_state, age FROM head ORDER BY age

 Mẫu 3

SQL TARGET: 
SELECT creation, name, budget_in_billions FROM department

 Mẫu 4

SQL TARGET: 
SELECT max(budget_in_billions), min(budget_in_billions) FROM department


In [6]:
from transformers import (Seq2SeqTrainer, Seq2SeqTrainingArguments)


training_args = Seq2SeqTrainingArguments(
    output_dir='.\\outputs\\t5_spider_text2sql',
    num_train_epochs=5,
    #Batch size
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    #tichs luy gradient,
    gradient_accumulation_steps=4,
    learning_rate=5e-5,
    weight_decay=0.01   ,
    warmup_steps=110,
    eval_strategy='epoch',
    save_strategy='epoch',
    logging_strategy='steps',
    logging_steps=50,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    report_to='none',
    fp16=False,
    dataloader_num_workers=0
)

trainer = Seq2SeqTrainer(model=model,args=training_args, train_dataset=tokenized_dataset,
                         eval_dataset=tokenizer_validation_set,
                         data_collator=data_collator)

In [7]:
#Kiểm tra thông số của laptop, có hỗ trợ cuda hay không
import torch 
import sys
print('CUDA available: ',torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU: ",torch.cuda.get_device_name(0))
    print(
        "VRAM:",
        round(
            torch.cuda.get_device_properties(0).total_memory/1024**3,2
        ),
        'GB'
    )
else: 
    print("Đang sử dụng CPU")
    import sys
import torch

print("Python:", sys.executable)
print("PyTorch version:", torch.__version__)
print("PyTorch CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("Torch location:", torch.__file__)

CUDA available:  True
GPU:  NVIDIA GeForce RTX 4050 Laptop GPU
VRAM: 6.0 GB
Python: d:\NLP_Projects\NLP_tieu_luan\.venv_NLP\Scripts\python.exe
PyTorch version: 2.14.1+cu126
PyTorch CUDA: 12.6
CUDA available: True
Torch location: d:\NLP_Projects\NLP_tieu_luan\.venv_NLP\Lib\site-packages\torch\__init__.py


In [8]:
#kiểm tra các thông số của trainer để fine tune model
print('Model: ',type(trainer.model))
print('Training samples: ',len(trainer.train_dataset))
print('Validation samples: ',len(trainer.eval_dataset))
print("Batch size: ",training_args.per_device_train_batch_size)
print('Device: ',training_args.device)

Model:  <class 'transformers.models.t5.modeling_t5.T5ForConditionalGeneration'>
Training samples:  7000
Validation samples:  1034
Batch size:  4
Device:  cuda:0


In [9]:
#kiểm tra trainer đã kết nối đúng với collator chưa
train_dataloader = trainer.get_train_dataloader()
print(train_dataloader)
first_batch = next(iter(train_dataloader))
print(first_batch)
for key, value in first_batch.items():
    print(key,value.shape)

{'input_ids': tensor([[11860,    10,   363,  ...,     0,     0,     0],
        [11860,    10,   363,  ...,   196,    26,     1],
        [11860,    10,   363,  ...,     0,     0,     0],
        [11860,    10,   571,  ...,     0,     0,     0]], device='cuda:0'), 'attention_mask': tensor([[1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0]], device='cuda:0'), 'labels': tensor([[    3, 23143, 14196,  5570, 21680,   690,  4674, 11300,   272,   476,
          5570,    71,  4112,     1,  -100,  -100,  -100,  -100,  -100,  -100,
          -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
          -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
          -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
          -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
          -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,

In [10]:
train_result = trainer.train()


Epoch,Training Loss,Validation Loss
1,4.798990,1.012272
2,3.424620,0.839358
3,2.913677,0.772824
4,2.718592,0.744901
5,2.652975,0.737559


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.66it/s]
[transformers] There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].


In [16]:
#Kiểm tra các chỉ số của bộ tham số được chọn trong model hiện tại
print("Best checkpoint: ",trainer.state.best_model_checkpoint)
print("Best validation loss: ",trainer.state.best_metric)
print('Global steps: ',trainer.state.global_step)


Best checkpoint:  .\outputs\t5_spider_text2sql\checkpoint-2190
Best validation loss:  0.7375594973564148
Global steps:  2190


In [17]:
#Kiểm tra khả năng sinh SQL của model sau fine tuning
from torch.utils.data import DataLoader

eval_subset = tokenizer_validation_set.select(range(10))
print(type(eval_subset))
eval_loader = DataLoader(eval_subset,batch_size=4,collate_fn=data_collator)
print(eval_loader)

<class 'datasets.arrow_dataset.Dataset'>


In [18]:
device = next(trainer.model.parameters()).device    
print(device)
trainer.model.eval()
all_predictions = []
all_targets = []

with torch.inference_mode():
    for batch in eval_loader:
        #chỉ lấy input_ids và attention_mask
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)

        #sinh SQL
        generated_ids = trainer.model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=256,
            num_beams=1
        )

        # Chuyển token ID dự đoán thành văn bản SQL
        predictions = tokenizer_t5.batch_decode(
            generated_ids,
            skip_special_tokens=True
        )

        #khôi phục labels: thay -100 bằng PAD   
        labels = batch['labels'].clone()
        labels[labels==-100] = tokenizer_t5.pad_token_id

        #chuyển labels thành SQL mục tiêu 
        targets = tokenizer_t5.batch_decode(
            labels.tolist(),
            skip_special_tokens = True
        )

        all_predictions.extend(predictions)
        all_targets.extend(targets)

cuda:0


In [14]:
for i in range(len(all_predictions)):
    print(f"\n MẪU {i + 1}")
    print("SQL TARGET: ")
    print(all_targets[i])

    print("\nSQL PREDICTED: ")
    print(all_predictions[i])


 MẪU 1
SQL TARGET: 
SELECT count(*) FROM singer

SQL PREDICTED: 
SELECT count(*) FROM singer

 MẪU 2
SQL TARGET: 
SELECT count(*) FROM singer

SQL PREDICTED: 
SELECT count(*) FROM singer

 MẪU 3
SQL TARGET: 
SELECT name, country, age FROM singer ORDER BY age DESC

SQL PREDICTED: 
SELECT Name, Country, age FROM singer WHERE age = "older" AND age = "older"

 MẪU 4
SQL TARGET: 
SELECT name, country, age FROM singer ORDER BY age DESC

SQL PREDICTED: 
SELECT T1.name, T2.country, T1.age FROM singer AS T1 JOIN singer AS T2 ON T1.singer_id = T2.singer_id WHERE T1.age = "abdescending"

 MẪU 5
SQL TARGET: 
SELECT avg(age), min(age), max(age) FROM singer WHERE country = 'France'

SQL PREDICTED: 
SELECT avg(age), avg(age), avg(age) FROM singer AS a t1 JOIN singer AS t2 ON t1.singer_id = t2.singer_id WHERE t1.country = "France" AND t2.age = "France"

 MẪU 6
SQL TARGET: 
SELECT avg(age), min(age), max(age) FROM singer WHERE country = 'France'

SQL PREDICTED: 
SELECT avg(age), avg(age), avg(age) FRO

In [15]:
trainer.save_model(
    "./outputs/t5_spider_text2sql/final"
)

tokenizer_t5.save_pretrained(
    "./outputs/t5_spider_text2sql/final"
)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.93it/s]


('./outputs/t5_spider_text2sql/final\\tokenizer_config.json',
 './outputs/t5_spider_text2sql/final\\tokenizer.json')